In [2]:
import cv2
import numpy as np

def compute_homography(image_path1, image_path2):
    """
    计算两幅图像之间的单应矩阵。

    参数：
        image_path1 (str): 第一幅图像的路径。
        image_path2 (str): 第二幅图像的路径。

    返回：
        H (numpy.ndarray): 计算得到的单应矩阵。
    """
    # 读取图像
    img1 = cv2.imread(image_path1, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(image_path2, cv2.IMREAD_GRAYSCALE)

    if img1 is None or img2 is None:
        raise FileNotFoundError("无法读取图像，请检查路径是否正确！")

    # 初始化 SIFT 特征检测器
    sift = cv2.SIFT_create()

    # 检测关键点和描述符
    keypoints1, descriptors1 = sift.detectAndCompute(img1, None)
    keypoints2, descriptors2 = sift.detectAndCompute(img2, None)

    # 使用 BFMatcher 进行特征匹配
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(descriptors1, descriptors2, k=2)

    # 应用比值测试来筛选匹配点
    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    if len(good_matches) < 4:
        raise ValueError("匹配点不足，无法计算单应矩阵！")

    # 提取匹配点的坐标
    src_pts = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # 计算单应矩阵
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    return H

# 示例调用
image_path1 = "./photos/2.jpg"  # 替换为第一幅图像的路径
image_path2 = "./photos/1.png"  # 替换为第二幅图像的路径

try:
    homography_matrix = compute_homography(image_path1, image_path2)
    print("计算得到的单应矩阵：")
    print(homography_matrix)
except Exception as e:
    print(f"发生错误: {e}")

计算得到的单应矩阵：
[[-5.46524209e+00  5.29220598e+00  3.68159507e+03]
 [-3.76448517e+00  4.26299146e+00  2.39931221e+03]
 [-1.37926160e-03  9.58442864e-04  1.00000000e+00]]
